# Denoising of corbel_big using CryoCARE "Even/Odd" 

* Inputs: `even.mrc` and `odd.mrc`.
* Outputs: denoised vol `even_odd_denoised/denoised.mrc`, and `denoised_even_odd.pdf` with a view of a tile of a central slice in Z.

In [1]:
from pathlib import Path

In [2]:
if Path("even_odd_denoised/vol.mrc").exists():
    raise Exception("even_odd_denoised/vol.mrc already exists ... exiting")

In [3]:
if not Path("even.mrc").exists() or not Path("odd.mrc"):
    %run split_even_odd.ipynb

noisy.shape=(100, 800, 800)
Writing even.mrc
noisy[0::2,:,:].shape=(50, 800, 800)
Writing odd.mrc
noisy[1::2,:,:].shape=(50, 800, 800)


In [4]:
from my_google_auth import DriveHandler
service = DriveHandler.get_drive_service()
handler = DriveHandler.DriveHandler(service)
MY_SHARED_DRIVE_ID = '1hGHvkP46fxLCQbUlyYhAS_eVl6PollQM'

An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'


/home/jupyter-vruiz/envs/cryoCARE/lib/python3.8/site-packages/google/api_core/_python_version_support.py:237: FutureWarning: You are using a non-supported Python version (3.8.19). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


## Configure cryoCARE

In [5]:
%%writefile train_data_config__evenodd.json
{
    "even": ["even.mrc"],
    "odd": ["odd.mrc"],
    "mask": [""],
    "patch_shape": [16, 16, 16],
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./even_odd_data",
    "overwrite": "True"  
}

Writing train_data_config__evenodd.json


In [6]:
%%bash
#cd /nas/vruiz/cryoCARE/empiar10311
source ~/envs/cryoCARE/bin/activate
cryoCARE_extract_train_data.py --conf train_data_config__evenodd.json

2026-03-18 20:18:10.007773: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
/home/jupyter-vruiz/envs/cryoCARE/lib/python3.8/site-packages/google/api_core/_python_version_support.py:237: FutureWarning: You are using a non-supported Python version (3.8.19). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'
even.data.shape=(50, 800, 800), sample_shape=[16, 16, 16]
Computing normalization parameters:


100%|██████████| 200/200 [00:00<00:00, 297.06it/s]


## Train

In [7]:
%%writefile train_config__evenodd.json
{
  "train_data": "./even_odd_data",
  "epochs": 50,
  "steps_per_epoch": 200,
  "batch_size": 16,
  "unet_kern_size": 3,
  "unet_n_depth": 3,
  "unet_n_first": 16,
  "learning_rate": 0.0004,
  "model_name": "model",
  "path": "./",
  "gpu_id": [1]
}

Writing train_config__evenodd.json


In [8]:
%%bash
#cd /nas/vruiz/cryoCARE/corbel_big
#pwd
source ~/envs/cryoCARE/bin/activate
#cryoCARE_extract_train_data.py --conf train_data_config__evenodd.json
cryoCARE_train.py --conf train_config__evenodd.json

2026-03-18 20:19:35.530143: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
2026-03-18 20:19:44.351384: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2026-03-18 20:19:44.366237: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2026-03-18 20:19:44.401283: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:941] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-03-18 20:19:44.402810: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:c1:00.0 name: NVIDIA A30 computeCapability: 8.0
coreClock: 1.44GHz coreCount: 56 deviceMemorySize: 23.60GiB deviceMemoryBandwidth: 869.04GiB/s
2026-03-18 20:19:44.402960: I tensorflow/stream_executor/cuda/cuda_gpu_executor.c

An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'
Looking for GPU with ID: 1
GPU 1 successfully found
0 1
1 16
2 16
3 16
4 1
Epoch 1/50


2026-03-18 20:19:58.859250: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudnn.so.8
2026-03-18 20:20:19.208655: W tensorflow/stream_executor/gpu/asm_compiler.cc:63] Running ptxas --version returned 256
2026-03-18 20:20:19.305046: W tensorflow/stream_executor/gpu/redzone_allocator.cc:314] Internal: ptxas exited with non-zero error code 256, output: 
Relying on driver to perform ptx compilation. 
Modify $PATH to customize ptxas location.
This message will be only logged once.
2026-03-18 20:20:20.340013: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.11
2026-03-18 20:20:24.163333: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublasLt.so.11
2026-03-18 20:20:58.219776: W tensorflow/core/grappler/optimizers/data/auto_shard.cc:656] In AUTO-mode, and switching to DATA-based sharding, instead of FILE-based sharding 

200/200 [==============================] - 81s 119ms/step - loss: 0.6218 - mse: 0.6218 - mae: 0.6219 - val_loss: 0.4457 - val_mse: 0.4457 - val_mae: 0.5306
Epoch 2/50
200/200 [==============================] - 3s 16ms/step - loss: 0.4466 - mse: 0.4466 - mae: 0.5306 - val_loss: 0.4252 - val_mse: 0.4252 - val_mae: 0.5180
Epoch 3/50
200/200 [==============================] - 3s 16ms/step - loss: 0.4311 - mse: 0.4311 - mae: 0.5215 - val_loss: 0.4242 - val_mse: 0.4242 - val_mae: 0.5173
Epoch 4/50
200/200 [==============================] - 3s 16ms/step - loss: 0.4274 - mse: 0.4274 - mae: 0.5194 - val_loss: 0.4250 - val_mse: 0.4250 - val_mae: 0.5167
Epoch 5/50
200/200 [==============================] - 3s 16ms/step - loss: 0.4250 - mse: 0.4250 - mae: 0.5179 - val_loss: 0.4190 - val_mse: 0.4190 - val_mae: 0.5147
Epoch 6/50
200/200 [==============================] - 3s 16ms/step - loss: 0.4246 - mse: 0.4246 - mae: 0.5176 - val_loss: 0.4182 - val_mse: 0.4182 - val_mae: 0.5136
Epoch 7/50
200/200 

In [ ]:
uploaded_file_id = handler.upload(
    local_file_path='model.tar.gz',
    drive_file_name='corbel_small__denoised__cryoCARE_even_odd__model.tar.gz',
    drive_folder_id=MY_SHARED_DRIVE_ID
)

## Infer

In [21]:
%%writefile predict_config__evenodd.json
{
    "path": "./model.tar.gz",
    "even": ["/home/jupyter-vruiz/gdrive_TomogramDenoising/tomograms/Corbel2301_block2_June2019_crop_ali_crop.mrc"], 
    "odd": ["/home/jupyter-vruiz/gdrive_TomogramDenoising/tomograms/Corbel2301_block2_June2019_crop_ali_crop.mrc"],
    "n_tiles": [1,1,1],
    "output": "even_odd_denoised",
    "overwrite": "True",
    "gpu_id": [1]
}

Overwriting predict_config__evenodd.json


In [22]:
%%bash
#cd /nas/vruiz/cryoCARE/corbel_big
pwd
source ~/envs/cryoCARE/bin/activate
cryoCARE_predict.py --conf predict_config__evenodd.json || true

/home/jupyter-vruiz/gdrive_TomogramDenoising/deenoising/docs/DAE/cryoCARE/corbel_small


2026-03-18 20:36:54.046935: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
2026-03-18 20:36:57.104198: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2026-03-18 20:36:57.104981: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2026-03-18 20:36:57.122050: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:941] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-03-18 20:36:57.123053: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:c1:00.0 name: NVIDIA A30 computeCapability: 8.0
coreClock: 1.44GHz coreCount: 56 deviceMemorySize: 23.60GiB deviceMemoryBandwidth: 869.04GiB/s
2026-03-18 20:36:57.123147: I tensorflow/stream_executor/cuda/cuda_gpu_executor.c

An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'
Looking for GPU with ID: 1
GPU 1 successfully found
Loading network weights from 'weights_best.h5'.
(104, 800, 800, 1)


Traceback (most recent call last):
  File "/home/jupyter-vruiz/envs/cryoCARE/bin/cryoCARE_predict.py", line 175, in <module>
    main()
  File "/home/jupyter-vruiz/envs/cryoCARE/bin/cryoCARE_predict.py", line 154, in main
    denoise(config, mean, std, even=even, odd=odd, output_file=out_filename)
  File "/home/jupyter-vruiz/envs/cryoCARE/bin/cryoCARE_predict.py", line 96, in denoise
    new_label = np.concatenate((even.header[l][1:-1], np.array([
  File "<__array_function__ internals>", line 5, in concatenate
UnicodeDecodeError: 'ascii' codec can't decode byte 0x84 in position 6: ordinal not in range(128)


In [27]:
!ls -l even_odd_denoised

total 250001
-rw-r--r-- 1 jupyter-vruiz jupyter-vruiz 256001024 mar 18 20:37 Corbel2301_block2_June2019_crop_ali_crop.mrc


In [ ]:
uploaded_file_id = handler.upload(
    local_file_path='even_odd_denoised/vol.mrc',
    drive_file_name='corbel_small__denoised__cryoCARE_even_odd.mrc',
    drive_folder_id=MY_SHARED_DRIVE_ID
)

In [23]:
import mrcfile
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

In [24]:
def read_MRC(file_path):
    return mrcfile.read(file_path)

In [26]:
mrc_file_path = 'even_odd_denoised/vol.m'
original_volume = read_MRC(mrc_file_path)

FileNotFoundError: [Errno 2] No such file or directory: 'even_odd_denoised/vol.mrc'

In [ ]:
original_volume.shape

In [ ]:
mrc_file_path = 'even_odd_denoised/vol.mrc'
denoised_volume = read_MRC(mrc_file_path)

In [ ]:
denoised_volume.shape

In [ ]:
# Choose a slice index in the middle of the volume for a good comparison
slice_idx = original_volume.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(20, 20))

# Plot the original slice z
im1 = axes[0].imshow(original_volume[slice_idx, :, :].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx}')
axes[0].grid(False)

# Plot the original slice z+1
im2 = axes[1].imshow(denoised_volume[slice_idx, :, :].T, cmap='gray', origin='lower')
axes[1].set_title(f'N2N Even/Odd Denoised Slice Z={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.pyplot import figure
figure(figsize=(16, 16))
slice_idx = denoised_volume.shape[0]//2
plt.imshow(denoised_volume[slice_idx, 0:400, 400:800], cmap="gray")
plt.savefig("denoised_even_odd.pdf", bbox_inches='tight')

In [ ]:
uploaded_file_id = handler.upload(
    local_file_path='denoised_even_odd.pdf',
    drive_file_name='corbel_small__denoised__cryoCARE_even_odd.pdf',
    drive_folder_id=MY_SHARED_DRIVE_ID
)